In [ ]:
# @title Inferensi video — satu sel
# @markdown Pilih **Runtime → Change runtime type → GPU**, isi formulir, lalu jalankan sel ini.
# @markdown Upload video ke panel Files, atau masukkan URL langsung ke berkas video untuk diunduh dengan **wget**.
video_url = "/content/test.mp4" # @param {type:"string"}
output_dir = "/content/outputs/check" # @param {type:"string"}
# @markdown Notebook menyertakan kode sendiri. Setelah git pull, buka ulang notebook terbaru di Colab.
# @markdown Folder output yang sama akan mengganti hasil sebelumnya.
# @markdown **Model** — kosongkan checkpoint lokal untuk unduh otomatis dari Hub.
model_repo = "azunzios/learning" # @param {type:"string"}
model_file = "model.pt" # @param {type:"string"}
local_checkpoint = "" # @param {type:"string"}
class_names_override = "" # @param {type:"string"}
# @markdown Jika metadata lama sudah netral, isi nama dipisahkan koma sesuai urutan head; contoh body_shaking,squirting.
# @markdown **Inferensi** — mulai dengan batch 8; turunkan ke 4/2/1 jika VRAM penuh.
batch_size = 8 # @param [1, 2, 4, 8, 16, 32] {type:"raw"}
stride_sec = 0.5 # @param {type:"number"}
# @markdown Stride 1.0/2.0 mengurangi jumlah jendela; detail waktu dan recall dapat turun.
prefetch = True # @param {type:"boolean"}
scan_mode = "full" # @param ["full", "adaptive"]
gate_ratio = 0.5 # @param {type:"number"}
refine_padding_sec = 2.0 # @param {type:"number"}
# @markdown Adaptive memindai kasar lalu memperinci kandidat. Bukan klasifikasi percakapan; kejadian singkat bisa terlewat.
analyze_from_sec = 0.0 # @param {type:"number"}
analyze_until_sec = -1.0 # @param {type:"number"}
# @markdown Batas akhir -1 berarti sampai akhir video; timestamp hasil mengikuti video asli.
use_fp16 = True # @param {type:"boolean"}
require_gpu = True # @param {type:"boolean"}
# @markdown Threshold **-1** memakai kalibrasi bawaan model.
event_1_threshold = -1.0 # @param {type:"number"}
event_2_threshold = -1.0 # @param {type:"number"}
smooth_sec = 1.0 # @param {type:"number"}
merge_gap_sec = 2.0 # @param {type:"number"}
min_duration_sec = 1.0 # @param {type:"number"}
low_ratio = 0.6 # @param {type:"number"}
# @markdown Simpan satu segmen dengan confidence peak tertinggi. Klip hasil tidak menyertakan audio.
preview_video = False # @param {type:"boolean"}
download_results = True # @param {type:"boolean"}

import importlib
import hashlib
import json
import math
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
from urllib.parse import urlsplit

if not video_url.strip():
    raise ValueError("Isi path video lokal atau URL langsung ke file video.")
if batch_size < 1 or batch_size != int(batch_size):
    raise ValueError("Batch harus bilangan bulat minimal 1.")
if not math.isfinite(stride_sec) or stride_sec <= 0:
    raise ValueError("Stride harus berupa angka positif.")
if (not math.isfinite(analyze_from_sec) or analyze_from_sec < 0
        or not math.isfinite(analyze_until_sec)
        or (analyze_until_sec != -1 and analyze_until_sec <= analyze_from_sec)):
    raise ValueError("Rentang analisis tidak valid.")
if not 0 < gate_ratio <= 1 or not math.isfinite(refine_padding_sec) or refine_padding_sec < 0:
    raise ValueError("Pengaturan adaptive tidak valid.")
if not 0 <= low_ratio <= 1 or any(not math.isfinite(v) or v < 0
                                 for v in (smooth_sec, merge_gap_sec, min_duration_sec)):
    raise ValueError("Pengaturan post-processing tidak valid.")
thresholds = {}
for index, value in enumerate((event_1_threshold, event_2_threshold), 1):
    if not math.isfinite(value) or (value != -1 and not 0 <= value <= 1):
        raise ValueError("Threshold harus -1 atau angka antara 0 dan 1.")
    if value != -1:
        thresholds[f"event_{index}"] = value

# Dependency dipasang otomatis; PyTorch bawaan Colab tetap digunakan jika sudah memenuhi versi.
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "torch>=2.1", "numpy",
    "opencv-python-headless", "matplotlib", "huggingface_hub",
], check=True)
import torch
if require_gpu and not torch.cuda.is_available():
    raise RuntimeError("Aktifkan runtime GPU, lalu jalankan ulang sel ini.")

# Paket inferensi disertakan di notebook sehingga tidak perlu clone repo.
package_sources = {'ved/__init__.py': '"""Video event detection & temporal localization (VideoMAEv2 + MIL)."""\n', 'ved/labels.py': '"""Checkpoint labels and backwards-compatible event aliases."""\nfrom __future__ import annotations\n\n\ndef inference_metadata(meta: dict, num_classes: int) -> dict:\n    """Retain the checkpoint\'s output order, labels and calibrated thresholds."""\n    result = neutral_metadata(meta, num_classes)\n    names = list(meta["class_names"])\n    result.update(class_names=names,\n                  display_names={name: meta.get("display_names", {}).get(name, name)\n                                 for name in names},\n                  thresholds={name: float(meta.get("thresholds", {}).get(name, 0.5))\n                              for name in names})\n    return result\n\n\ndef neutral_metadata(meta: dict, num_classes: int) -> dict:\n    """Keep output order and calibrated thresholds from an existing export.\n\n    Only inference settings are retained. Historical training configuration and\n    display text are intentionally omitted from the returned metadata.\n    """\n    previous = list(meta["class_names"])\n    if num_classes < 1 or len(previous) != num_classes or len(set(previous)) != num_classes:\n        raise ValueError("Checkpoint class metadata does not match the output head.")\n    names = [f"event_{i + 1}" for i in range(num_classes)]\n    thresholds = meta.get("thresholds", {})\n    return {\n        "class_names": names,\n        "display_names": {name: f"Event {i + 1}" for i, name in enumerate(names)},\n        "thresholds": {name: float(thresholds.get(old, 0.5))\n                       for old, name in zip(previous, names)},\n        "fps": float(meta["fps"]),\n        "num_frames": int(meta["num_frames"]),\n        "img_size": int(meta["img_size"]),\n    }\n', 'ved/model.py': '"""VideoMAEv2 ViT backbone (compatible with the official OpenGVLab checkpoints)\nplus a small multi-label classification head.\n\nThe official distilled checkpoint `distill/vit_s_k710_dl_from_giant.pth`\n(ViT-Small, 16 frames, 224px, tubelet 2, patch 16) loads into this module.\nThe architecture is inferred from the checkpoint itself, so ViT-Base\ncheckpoints also work.\n"""\nfrom __future__ import annotations\n\nimport math\nfrom functools import partial\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom .labels import inference_metadata\n\n\n# --------------------------------------------------------------------------- #\n# Building blocks\n# --------------------------------------------------------------------------- #\nclass DropPath(nn.Module):\n    def __init__(self, p: float = 0.0):\n        super().__init__()\n        self.p = p\n\n    def forward(self, x):\n        if self.p == 0.0 or not self.training:\n            return x\n        keep = 1 - self.p\n        mask = x.new_empty((x.shape[0],) + (1,) * (x.ndim - 1)).bernoulli_(keep)\n        return x * mask / keep\n\n\nclass Mlp(nn.Module):\n    def __init__(self, dim, hidden):\n        super().__init__()\n        self.fc1 = nn.Linear(dim, hidden)\n        self.act = nn.GELU()\n        self.fc2 = nn.Linear(hidden, dim)\n\n    def forward(self, x):\n        return self.fc2(self.act(self.fc1(x)))\n\n\nclass Attention(nn.Module):\n    def __init__(self, dim, num_heads, qkv_bias=True):\n        super().__init__()\n        self.num_heads = num_heads\n        self.qkv = nn.Linear(dim, dim * 3, bias=False)\n        if qkv_bias:\n            self.q_bias = nn.Parameter(torch.zeros(dim))\n            self.v_bias = nn.Parameter(torch.zeros(dim))\n        else:\n            self.q_bias = self.v_bias = None\n        self.proj = nn.Linear(dim, dim)\n\n    def forward(self, x):\n        B, N, C = x.shape\n        bias = None\n        if self.q_bias is not None:\n            bias = torch.cat([self.q_bias, torch.zeros_like(self.v_bias), self.v_bias])\n        qkv = F.linear(x, self.qkv.weight, bias)\n        qkv = qkv.reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)\n        q, k, v = qkv[0], qkv[1], qkv[2]\n        x = F.scaled_dot_product_attention(q, k, v)\n        x = x.transpose(1, 2).reshape(B, N, C)\n        return self.proj(x)\n\n\nclass Block(nn.Module):\n    def __init__(self, dim, num_heads, mlp_ratio=4.0, drop_path=0.0, init_values=0.0):\n        super().__init__()\n        norm = partial(nn.LayerNorm, eps=1e-6)\n        self.norm1 = norm(dim)\n        self.attn = Attention(dim, num_heads)\n        self.drop_path = DropPath(drop_path)\n        self.norm2 = norm(dim)\n        self.mlp = Mlp(dim, int(dim * mlp_ratio))\n        if init_values > 0:\n            self.gamma_1 = nn.Parameter(init_values * torch.ones(dim))\n            self.gamma_2 = nn.Parameter(init_values * torch.ones(dim))\n        else:\n            self.gamma_1 = self.gamma_2 = None\n\n    def forward(self, x):\n        if self.gamma_1 is None:\n            x = x + self.drop_path(self.attn(self.norm1(x)))\n            x = x + self.drop_path(self.mlp(self.norm2(x)))\n        else:\n            x = x + self.drop_path(self.gamma_1 * self.attn(self.norm1(x)))\n            x = x + self.drop_path(self.gamma_2 * self.mlp(self.norm2(x)))\n        return x\n\n\ndef sinusoid_table(n_position, d_hid):\n    def angle_vec(pos):\n        return [pos / np.power(10000, 2 * (j // 2) / d_hid) for j in range(d_hid)]\n\n    table = np.array([angle_vec(p) for p in range(n_position)])\n    table[:, 0::2] = np.sin(table[:, 0::2])\n    table[:, 1::2] = np.cos(table[:, 1::2])\n    return torch.tensor(table, dtype=torch.float32).unsqueeze(0)\n\n\nclass VideoViT(nn.Module):\n    """VideoMAEv2 encoder with mean pooling. Input: (B, 3, T, H, W)."""\n\n    def __init__(self, img_size=224, patch_size=16, tubelet_size=2, num_frames=16,\n                 embed_dim=384, depth=12, num_heads=6, mlp_ratio=4.0,\n                 drop_path_rate=0.1, init_values=0.0):\n        super().__init__()\n        self.embed_dim = embed_dim\n        self.depth = depth\n        self.patch_embed = nn.Module()\n        self.patch_embed.proj = nn.Conv3d(\n            3, embed_dim,\n            kernel_size=(tubelet_size, patch_size, patch_size),\n            stride=(tubelet_size, patch_size, patch_size))\n        n_patches = (img_size // patch_size) ** 2 * (num_frames // tubelet_size)\n        self.register_buffer("pos_embed", sinusoid_table(n_patches, embed_dim), persistent=True)\n        dpr = torch.linspace(0, drop_path_rate, depth).tolist()\n        self.blocks = nn.ModuleList([\n            Block(embed_dim, num_heads, mlp_ratio, dpr[i], init_values) for i in range(depth)])\n        self.fc_norm = nn.LayerNorm(embed_dim, eps=1e-6)\n\n    def forward(self, x):\n        x = self.patch_embed.proj(x).flatten(2).transpose(1, 2)\n        x = x + self.pos_embed.to(x.dtype)\n        for blk in self.blocks:\n            x = blk(x)\n        return self.fc_norm(x.mean(1))\n\n\nclass EventClassifier(nn.Module):\n    """Backbone + multi-label head (one sigmoid output per event class)."""\n\n    def __init__(self, backbone_cfg: dict, num_classes: int, head_dropout: float = 0.3):\n        super().__init__()\n        self.backbone_cfg = dict(backbone_cfg)\n        self.backbone = VideoViT(**backbone_cfg)\n        self.dropout = nn.Dropout(head_dropout)\n        self.head = nn.Linear(self.backbone.embed_dim, num_classes)\n        nn.init.trunc_normal_(self.head.weight, std=0.02)\n        nn.init.zeros_(self.head.bias)\n\n    def forward(self, x):\n        return self.head(self.dropout(self.backbone(x)))\n\n\n# --------------------------------------------------------------------------- #\n# Checkpoint helpers\n# --------------------------------------------------------------------------- #\ndef _clean_state_dict(ckpt):\n    for key in ("model", "module", "state_dict"):\n        if isinstance(ckpt, dict) and key in ckpt and isinstance(ckpt[key], dict):\n            ckpt = ckpt[key]\n    out = {}\n    for k, v in ckpt.items():\n        for prefix in ("module.", "backbone.", "encoder."):\n            if k.startswith(prefix):\n                k = k[len(prefix):]\n        out[k] = v\n    # checkpoints without mean pooling store the final norm as `norm.*`\n    if "fc_norm.weight" not in out and "norm.weight" in out:\n        out["fc_norm.weight"] = out.pop("norm.weight")\n        out["fc_norm.bias"] = out.pop("norm.bias")\n    return out\n\n\ndef infer_backbone_cfg(sd: dict, num_frames=16, img_size=224, drop_path_rate=0.1) -> dict:\n    w = sd["patch_embed.proj.weight"]  # (D, 3, t, p, p)\n    embed_dim, _, tubelet, patch, _ = w.shape\n    depth = 1 + max(int(k.split(".")[1]) for k in sd if k.startswith("blocks."))\n    mlp_hidden = sd["blocks.0.mlp.fc1.weight"].shape[0]\n    return dict(\n        img_size=img_size, patch_size=int(patch), tubelet_size=int(tubelet),\n        num_frames=num_frames, embed_dim=int(embed_dim), depth=int(depth),\n        num_heads=int(embed_dim) // 64, mlp_ratio=mlp_hidden / embed_dim,\n        drop_path_rate=drop_path_rate,\n        init_values=0.1 if "blocks.0.gamma_1" in sd else 0.0,\n    )\n\n\ndef build_from_pretrained(path: str, num_classes: int, num_frames=16, img_size=224,\n                          drop_path_rate=0.1, head_dropout=0.3) -> EventClassifier:\n    ckpt = torch.load(path, map_location="cpu", weights_only=False)\n    sd = _clean_state_dict(ckpt)\n    cfg = infer_backbone_cfg(sd, num_frames, img_size, drop_path_rate)\n    model = EventClassifier(cfg, num_classes, head_dropout)\n    bb_sd = {k: v for k, v in sd.items() if not k.startswith("head.")}\n    if "pos_embed" in bb_sd and bb_sd["pos_embed"].shape != model.backbone.pos_embed.shape:\n        bb_sd.pop("pos_embed")\n    missing, unexpected = model.backbone.load_state_dict(bb_sd, strict=False)\n    missing = [k for k in missing if k != "pos_embed"]\n    print(f"[pretrained] backbone cfg: {cfg}")\n    print(f"[pretrained] missing keys: {missing}")\n    print(f"[pretrained] unexpected keys: {unexpected}")\n    if any(k.startswith("blocks.") or k.startswith("patch_embed") for k in missing):\n        raise RuntimeError("Backbone weights did not load correctly - check the checkpoint.")\n    return model\n\n\ndef save_export(model: EventClassifier, path: str, meta: dict):\n    sd = {k: v.half() if v.is_floating_point() else v for k, v in model.state_dict().items()}\n    cfg = dict(model.backbone_cfg)\n    cfg["drop_path_rate"] = 0.0\n    torch.save({"state_dict": sd, "backbone_cfg": cfg,\n                "num_classes": model.head.out_features,\n                "meta": inference_metadata(meta, model.head.out_features)}, path)\n\n\ndef load_export(path: str, device="cpu"):\n    ckpt = torch.load(path, map_location="cpu", weights_only=True)\n    model = EventClassifier(ckpt["backbone_cfg"], ckpt["num_classes"], head_dropout=0.0)\n    model.load_state_dict({k: v.float() for k, v in ckpt["state_dict"].items()})\n    meta = inference_metadata(ckpt["meta"], ckpt["num_classes"])\n    model.output_labels = dict(zip(meta["class_names"], meta["class_names"]))\n    model.output_display = dict(ckpt["meta"].get("display_names", {}))\n    return model.to(device).eval(), meta\n', 'ved/video_io.py': '"""Video decoding: sample frames at a fixed FPS and resize them to a square."""\nfrom __future__ import annotations\n\nfrom typing import Iterator, Tuple\nimport math\nimport os\nfrom pathlib import Path\nimport tempfile\n\nimport cv2\nimport numpy as np\n\nVIDEO_EXTS = {".mp4", ".mkv", ".avi", ".mov", ".webm", ".m4v", ".wmv", ".flv", ".mpg", ".mpeg", ".ts"}\n\n\ndef video_info(path: str) -> Tuple[float, int, float]:\n    cap = cv2.VideoCapture(path)\n    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0\n    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)\n    cap.release()\n    if fps <= 1e-3 or fps > 240:\n        fps = 30.0\n    return fps, n, (n / fps if n > 0 else 0.0)\n\n\ndef iter_frames(path: str, target_fps: float, size: int,\n                start_sec: float = 0, end_sec: float | None = None) -> Iterator[np.ndarray]:\n    """Yield RGB uint8 frames (size x size x 3) sampled at `target_fps`.\n\n    Frame k corresponds to timestamp k / target_fps seconds.\n    Frames are squashed (not cropped) to keep the whole scene visible.\n    """\n    if not math.isfinite(start_sec) or start_sec < 0 or (end_sec is not None and\n            (not math.isfinite(end_sec) or end_sec <= start_sec)):\n        raise ValueError("Invalid decoding interval.")\n    cap = cv2.VideoCapture(path)\n    if not cap.isOpened():\n        raise IOError(f"Cannot open video: {path}")\n    src_fps = cap.get(cv2.CAP_PROP_FPS) or 0.0\n    if src_fps <= 1e-3 or src_fps > 240:\n        src_fps = 30.0\n    step = 1.0 / target_fps\n    next_t = start_sec\n    idx = max(0, int(math.ceil(start_sec * src_fps - 1e-6)))\n    try:\n        if idx and not cap.set(cv2.CAP_PROP_POS_FRAMES, idx):\n            raise IOError("Video backend cannot seek to the requested interval.")\n        while True:\n            t = idx / src_fps\n            if end_sec is not None and next_t >= end_sec - 1e-6:\n                break\n            ok = cap.grab()\n            if not ok:\n                break\n            idx += 1\n            if t + 1e-6 < next_t:\n                continue\n            ok, frame = cap.retrieve()\n            if not ok:\n                break\n            frame = cv2.resize(frame, (size, size), interpolation=cv2.INTER_AREA)\n            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)\n            # Repeat frames if the source FPS is below the model FPS.\n            while next_t <= t + 1e-6 and (end_sec is None or next_t < end_sec - 1e-6):\n                yield frame\n                next_t += step\n    finally:\n        cap.release()\n\n\ndef read_frames(path: str, target_fps: float, size: int) -> np.ndarray:\n    frames = list(iter_frames(path, target_fps, size))\n    if not frames:\n        raise IOError(f"No frames decoded from: {path}")\n    return np.stack(frames)\n\n\ndef export_segment(video_path: str, output_path: str, start: float, end: float) -> str:\n    """Save one detected interval using OpenCV (video only, without audio)."""\n    if not math.isfinite(start) or not math.isfinite(end) or start < 0 or end <= start:\n        raise ValueError("Invalid clip interval.")\n    output = Path(output_path).resolve()\n    if output == Path(video_path).resolve():\n        raise ValueError("The clip output must differ from the source video.")\n    output.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.TemporaryDirectory(prefix=".clip-", dir=output.parent) as temporary:\n        destination = Path(temporary) / "clip.mp4"\n        cap = cv2.VideoCapture(video_path)\n        writer = None\n        written = 0\n        try:\n            if not cap.isOpened():\n                raise IOError("Cannot open the source video for clipping.")\n            fps = cap.get(cv2.CAP_PROP_FPS)\n            if not math.isfinite(fps) or fps <= 0:\n                raise ValueError("Cannot determine source video FPS.")\n            width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))\n            height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))\n            first, last = int(math.floor(start * fps)), int(math.ceil(end * fps))\n            cap.set(cv2.CAP_PROP_POS_FRAMES, first)\n            writer = cv2.VideoWriter(str(destination), cv2.VideoWriter_fourcc(*"mp4v"),\n                                     fps, (width, height))\n            if not writer.isOpened():\n                raise IOError("Cannot create the output MP4.")\n            for _ in range(first, last):\n                ok, frame = cap.read()\n                if not ok:\n                    break\n                writer.write(frame)\n                written += 1\n        finally:\n            cap.release()\n            if writer is not None:\n                writer.release()\n        if not written or not destination.is_file() or destination.stat().st_size == 0:\n            raise IOError("The selected interval produced no video frames.")\n        os.replace(destination, output)\n    return str(output)\n', 'ved/postprocess.py': '"""Turn sliding-window probabilities into merged time segments.\n\nPipeline (per class):\n  1. Overlap-add: every frame gets the mean (or max) probability of all\n     windows that cover it  -> a frame-level score curve.\n  2. Temporal smoothing (moving average) to remove single-window spikes.\n  3. Hysteresis thresholding: a segment is a connected region where\n     score >= low that contains at least one frame with score >= high.\n     This keeps boundaries stable without flickering on/off.\n  4. Gap merging: segments of the same class separated by <= merge_gap\n     seconds are joined (the event was briefly occluded / low score).\n  5. Minimum duration filter: drop segments shorter than min_dur seconds.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import asdict, dataclass\nfrom typing import Dict, List, Sequence\n\nimport numpy as np\n\n\n@dataclass\nclass Segment:\n    label: str\n    start: float\n    end: float\n    peak: float\n    mean: float\n\n    @property\n    def duration(self):\n        return self.end - self.start\n\n    def to_dict(self):\n        d = asdict(self)\n        d["duration"] = round(self.duration, 3)\n        return d\n\n\ndef windows_to_frame_scores(starts: Sequence[int], probs: np.ndarray, num_frames_total: int,\n                            window_len: int, agg: str = "mean") -> np.ndarray:\n    """starts: window start frame indices; probs: (num_windows, C)."""\n    C = probs.shape[1]\n    T = max(num_frames_total, 1)\n    if agg == "max":\n        out = np.zeros((T, C), dtype=np.float32)\n        for s, p in zip(starts, probs):\n            e = min(s + window_len, T)\n            out[s:e] = np.maximum(out[s:e], p)\n        return out\n    acc = np.zeros((T, C), dtype=np.float64)\n    cnt = np.zeros((T, 1), dtype=np.float64)\n    for s, p in zip(starts, probs):\n        e = min(s + window_len, T)\n        acc[s:e] += p\n        cnt[s:e] += 1\n    cnt[cnt == 0] = 1\n    return (acc / cnt).astype(np.float32)\n\n\ndef smooth(scores: np.ndarray, k: int) -> np.ndarray:\n    if k <= 1:\n        return scores\n    kernel = np.ones(k) / k\n    pad = k // 2\n    out = np.empty_like(scores)\n    for c in range(scores.shape[1]):\n        x = np.pad(scores[:, c], (pad, k - 1 - pad), mode="edge")\n        out[:, c] = np.convolve(x, kernel, mode="valid")\n    return out\n\n\ndef hysteresis_regions(s: np.ndarray, high: float, low: float):\n    above = s >= low\n    regions, i, n = [], 0, len(s)\n    while i < n:\n        if above[i]:\n            j = i\n            while j < n and above[j]:\n                j += 1\n            if s[i:j].max() >= high:\n                regions.append((i, j))\n            i = j\n        else:\n            i += 1\n    return regions\n\n\ndef merge_regions(regions, max_gap_frames: int):\n    if not regions:\n        return []\n    merged = [list(regions[0])]\n    for s, e in regions[1:]:\n        if s - merged[-1][1] <= max_gap_frames:\n            merged[-1][1] = e\n        else:\n            merged.append([s, e])\n    return [tuple(r) for r in merged]\n\n\ndef frame_scores_to_segments(scores: np.ndarray, fps: float, class_names: List[str],\n                             thresholds: Dict[str, float], low_ratio: float = 0.6,\n                             smooth_sec: float = 1.0, merge_gap: float = 2.0,\n                             min_dur: float = 1.0) -> List[Segment]:\n    sm = smooth(scores, max(1, int(round(smooth_sec * fps))))\n    segs: List[Segment] = []\n    for c, name in enumerate(class_names):\n        high = float(thresholds.get(name, 0.5))\n        low = high * low_ratio\n        regions = hysteresis_regions(sm[:, c], high, low)\n        regions = merge_regions(regions, int(round(merge_gap * fps)))\n        for s, e in regions:\n            if (e - s) / fps < min_dur:\n                continue\n            seg_scores = sm[s:e, c]\n            segs.append(Segment(name, round(s / fps, 2), round(e / fps, 2),\n                                round(float(seg_scores.max()), 3), round(float(seg_scores.mean()), 3)))\n    segs.sort(key=lambda x: (x.start, x.label))\n    return segs\n\n\ndef fmt_time(sec: float) -> str:\n    m, s = divmod(sec, 60)\n    h, m = divmod(int(m), 60)\n    return f"{h:02d}:{m:02d}:{s:05.2f}" if h else f"{m:02d}:{s:05.2f}"\n\n\ndef format_segments(segs: List[Segment], display: Dict[str, str] | None = None) -> str:\n    if not segs:\n        return "No events detected."\n    lines = []\n    for s in segs:\n        name = (display or {}).get(s.label, s.label)\n        lines.append(f"[{fmt_time(s.start)} -> {fmt_time(s.end)}] Detected {name} "\n                     f"from {s.start:.2f}s until {s.end:.2f}s "\n                     f"(duration {s.duration:.2f}s, peak {s.peak:.2f}, mean {s.mean:.2f})")\n    return "\\n".join(lines)\n', 'ved/inference.py': '"""Sliding-window inference over long videos (runs fine on an RTX 4070, 12 GB)."""\nfrom __future__ import annotations\n\nimport json\nimport math\nimport time\nfrom collections import deque\nfrom concurrent.futures import ThreadPoolExecutor\nfrom typing import Dict, Optional\n\nimport numpy as np\nimport torch\n\nfrom .model import load_export\nfrom .postprocess import format_segments, frame_scores_to_segments, windows_to_frame_scores\nfrom .video_io import iter_frames, video_info\n\nMEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1, 1)\nSTD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1, 1)\n\n\ndef to_model_input(batch_uint8: torch.Tensor) -> torch.Tensor:\n    """(B, T, H, W, 3) uint8 -> (B, 3, T, H, W) normalized float."""\n    x = batch_uint8.permute(0, 4, 1, 2, 3).float().div_(255.0)\n    return (x - MEAN.to(x.device)) / STD.to(x.device)\n\n\nclass Detector:\n    def __init__(self, ckpt_path: str, device: Optional[str] = None, fp16: bool = True,\n                 class_names=None):\n        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")\n        if self.device.startswith("cuda") and not torch.cuda.is_available():\n            raise ValueError("CUDA is unavailable in this Python environment. Install a CUDA PyTorch build or use --device cpu.")\n        self.model, self.meta = load_export(ckpt_path, self.device)\n        if class_names is not None:\n            previous = self.meta["class_names"]\n            names = list(class_names)\n            if len(names) != len(previous) or len(set(names)) != len(names) or any(not n.strip() for n in names):\n                raise ValueError("class_names must contain one unique name per model output, in checkpoint order.")\n            self.meta["thresholds"] = dict(zip(names, (self.meta["thresholds"][n] for n in previous)))\n            self.meta["class_names"] = names\n            self.meta["display_names"] = dict(zip(names, names))\n            self.model.output_labels = dict(zip(names, names))\n            self.model.output_display = dict(zip(names, names))\n        self.fp16 = fp16 and self.device.startswith("cuda")\n        if self.fp16:\n            self.model.half()\n        self.class_names = self.meta["class_names"]\n        self.display = self.meta.get("display_names", {})\n        self.fps = float(self.meta["fps"])\n        self.win = int(self.meta["num_frames"])\n        self.size = int(self.meta["img_size"])\n        self.thresholds = dict(self.meta.get("thresholds", {}))\n        self._mean = MEAN.to(self.device)\n        self._std = STD.to(self.device)\n        self.last_stats = {}\n\n    def resolve_thresholds(self, thresholds=None):\n        resolved = {}\n        aliases = {f"event_{i + 1}": name for i, name in enumerate(self.class_names)}\n        for key, value in (thresholds or {}).items():\n            name = key if key in self.class_names else aliases.get(key)\n            if name is None:\n                raise ValueError(f"Unknown class {key!r}; use {\', \'.join(self.class_names)} or event_N aliases.")\n            if not math.isfinite(value) or not 0 <= value <= 1:\n                raise ValueError("Thresholds must be finite numbers between 0 and 1.")\n            if name in resolved and resolved[name] != value:\n                raise ValueError(f"Conflicting thresholds for {name}.")\n            resolved[name] = value\n        return self.thresholds | resolved\n\n    @torch.inference_mode()\n    def _run(self, windows):\n        x = torch.from_numpy(np.stack(windows)).to(self.device, non_blocking=True)\n        x = x.permute(0, 4, 1, 2, 3).float()\n        x.div_(255.0).sub_(self._mean).div_(self._std)\n        if self.fp16:\n            x = x.half()\n        return torch.sigmoid(self.model(x).float()).cpu().numpy()\n\n    def _batches(self, video_path, stride, batch_size, first, end_sec):\n        """Decode on one producer; retain at most one batch ahead of inference."""\n        buf = deque(maxlen=self.win)\n        pending, starts = [], []\n        n, last_start = 0, -1\n        frames = iter_frames(video_path, self.fps, self.size, first / self.fps, end_sec)\n        try:\n            for frame in frames:\n                buf.append(frame)\n                n += 1\n                s = n - self.win\n                if s >= 0 and s % stride == 0:\n                    pending.append(np.stack(buf))\n                    starts.append(first + s)\n                    last_start = s\n                if len(pending) >= batch_size:\n                    yield pending, starts, first + n\n                    pending, starts = [], []\n            if n == 0:\n                raise IOError(f"No frames decoded from {video_path} in the selected interval")\n            if n < self.win:\n                pending.append(np.stack(list(buf) + [buf[-1]] * (self.win - n)))\n                starts.append(first)\n            elif last_start != n - self.win:\n                pending.append(np.stack(buf))\n                starts.append(first + n - self.win)\n            if pending:\n                yield pending, starts, first + n\n        finally:\n            frames.close()\n\n    def predict_windows(self, video_path: str, stride_sec: float = 0.5, batch_size: int = 8,\n                        verbose: bool = True, prefetch: bool = True,\n                        start_sec: float = 0, end_sec: float | None = None):\n        if not math.isfinite(batch_size) or batch_size < 1 or int(batch_size) != batch_size:\n            raise ValueError("batch_size must be a positive integer.")\n        if not math.isfinite(stride_sec) or not 0 < stride_sec <= self.win / self.fps:\n            raise ValueError("stride_sec must be positive and no longer than one model window.")\n        stride = max(1, int(round(stride_sec * self.fps)))\n        if not math.isfinite(start_sec) or start_sec < 0 or (end_sec is not None and\n                (not math.isfinite(end_sec) or end_sec <= start_sec)):\n            raise ValueError("Invalid inference interval.")\n        first = int(math.ceil(start_sec * self.fps - 1e-6))\n        starts, probs = [], []\n        n = first\n        _, _, dur = video_info(video_path)\n        t0 = time.perf_counter()\n        batches = self._batches(video_path, stride, batch_size, first, end_sec)\n        executor = ThreadPoolExecutor(max_workers=1) if prefetch else None\n        future = executor.submit(next, batches, None) if executor else None\n        try:\n            while True:\n                item = future.result() if executor else next(batches, None)\n                if item is None:\n                    break\n                if executor:\n                    future = executor.submit(next, batches, None)\n                pending, pending_starts, n = item\n                probs.append(self._run(pending))\n                starts.extend(pending_starts)\n                if verbose and dur > 0:\n                    print(f"\\r  processed {(n - first) / self.fps:7.1f}s "\n                          f"({((n - first) / self.fps) / max(time.perf_counter() - t0, 1e-6):.1f}x realtime)",\n                          end="", flush=True)\n        finally:\n            if executor:\n                future.cancel()\n                executor.shutdown(wait=True, cancel_futures=True)\n            batches.close()\n        elapsed = time.perf_counter() - t0\n        self.last_stats = {"seconds": round(elapsed, 4), "windows": len(starts),\n                           "video_seconds": (n - first) / self.fps,\n                           "realtime_factor": round((n - first) / self.fps / max(elapsed, 1e-6), 3)}\n        if verbose:\n            print(f"\\r  done: {(n - first) / self.fps:.1f}s in {elapsed:.2f}s "\n                  f"({self.last_stats[\'realtime_factor\']:.1f}x realtime, {len(starts)} windows)" + " " * 10)\n        return np.array(starts), np.concatenate(probs, 0), n\n\n    def detect(self, video_path: str, stride_sec: float = 0.5, agg: str = "mean",\n               thresholds: Optional[Dict[str, float]] = None, low_ratio: float = 0.6,\n               smooth_sec: float = 1.0, merge_gap: float = 2.0, min_dur: float = 1.0,\n               batch_size: int = 8, verbose: bool = True, prefetch: bool = True,\n               start_sec: float = 0, end_sec: float | None = None,\n               scan: str = "full", gate_ratio: float = 0.5, refine_padding: float = 2.0):\n        thr = self.resolve_thresholds(thresholds)\n        if agg not in {"mean", "max"} or scan not in {"full", "adaptive"}:\n            raise ValueError("Use agg=mean/max and scan=full/adaptive.")\n        if not 0 <= low_ratio <= 1 or any(not math.isfinite(v) or v < 0\n                                         for v in (smooth_sec, merge_gap, min_dur, refine_padding)):\n            raise ValueError("Invalid post-processing or padding settings.")\n        if not math.isfinite(gate_ratio) or not 0 < gate_ratio <= 1:\n            raise ValueError("gate_ratio must be greater than 0 and at most 1.")\n        if not math.isfinite(stride_sec) or not 0 < stride_sec <= self.win / self.fps:\n            raise ValueError("stride_sec must be positive and no longer than one model window.")\n        t0 = time.perf_counter()\n        first = int(math.ceil(start_sec * self.fps - 1e-6)) if math.isfinite(start_sec) else 0\n        coarse_stride = self.win / self.fps if scan == "adaptive" else stride_sec\n        starts, probs, n = self.predict_windows(video_path, coarse_stride, batch_size, verbose,\n                                                prefetch, start_sec, end_sec)\n        predicted_windows = len(starts)\n        refined_ranges = []\n        if scan == "adaptive" and max(1, int(round(stride_sec * self.fps))) < self.win:\n            # This gates on existing event scores; it is not a separate scene classifier.\n            gate = np.array([thr[name] * gate_ratio for name in self.class_names])\n            pad = self.win + int(math.ceil(refine_padding * self.fps))\n            for s in starts[np.any(probs >= gate, axis=1)]:\n                lo, hi = max(first, int(s) - pad), min(n, int(s) + self.win + pad)\n                if refined_ranges and lo <= refined_ranges[-1][1]:\n                    refined_ranges[-1][1] = max(refined_ranges[-1][1], hi)\n                else:\n                    refined_ranges.append([lo, hi])\n            parts_s, parts_p = [starts], [probs]\n            for lo, hi in refined_ranges:\n                refined_s, refined_p, _ = self.predict_windows(\n                    video_path, stride_sec, batch_size, verbose, prefetch, lo / self.fps, hi / self.fps)\n                predicted_windows += len(refined_s)\n                parts_s.append(refined_s)\n                parts_p.append(refined_p)\n            all_s, all_p = np.concatenate(parts_s), np.concatenate(parts_p)\n            # Prefer refined predictions for duplicate windows; never weight duplicates twice.\n            _, indices = np.unique(all_s[::-1], return_index=True)\n            indices = len(all_s) - 1 - indices\n            order = indices[np.argsort(all_s[indices])]\n            starts, probs = all_s[order], all_p[order]\n        frame_scores = windows_to_frame_scores(starts, probs, n, self.win, agg)\n        # Restrict post-processing to the requested interval, including threshold=0.\n        segs = frame_scores_to_segments(frame_scores[first:n], self.fps, self.class_names, thr,\n                                        low_ratio, smooth_sec, merge_gap, min_dur)\n        for segment in segs:\n            segment.start = round(segment.start + first / self.fps, 3)\n            segment.end = round(segment.end + first / self.fps, 3)\n        elapsed = time.perf_counter() - t0\n        stats = {"scan": scan, "seconds": round(elapsed, 4), "windows": predicted_windows,\n                 "unique_windows": len(starts), "video_seconds": (n - first) / self.fps,\n                 "realtime_factor": round((n - first) / self.fps / max(elapsed, 1e-6), 3),\n                 "analyzed_range": [first / self.fps, n / self.fps],\n                 "refined_ranges": [[lo / self.fps, hi / self.fps] for lo, hi in refined_ranges],\n                 "device": self.device, "fp16": self.fp16, "batch_size": batch_size,\n                 "stride_sec": stride_sec, "prefetch": prefetch,\n                 "gate_ratio": gate_ratio if scan == "adaptive" else None,\n                 "refine_padding": refine_padding if scan == "adaptive" else None,\n                 "agg": agg, "smooth_sec": smooth_sec, "merge_gap": merge_gap,\n                 "min_duration": min_dur, "low_ratio": low_ratio}\n        if verbose:\n            print(f"Inference total: {elapsed:.2f}s, {stats[\'realtime_factor\']:.1f}x realtime, "\n                  f"{predicted_windows} windows ({scan})")\n        return {"video": video_path, "segments": segs, "frame_scores": frame_scores,\n                "fps": self.fps, "thresholds": thr, "performance": stats,\n                "output_labels": getattr(self.model, "output_labels", {}),\n                "output_display": getattr(self.model, "output_display", {})}\n\n    def report(self, result) -> str:\n        return format_segments(result["segments"], self.display)\n\n\ndef save_result(result, json_path: str, display: Optional[Dict[str, str]] = None):\n    labels = result.get("output_labels", {})\n    original_display = result.get("output_display", {})\n    segments = []\n    for segment in result["segments"]:\n        name = labels.get(segment.label, segment.label)\n        segments.append(dict(segment.to_dict(), label=name,\n                             display_name=original_display.get(name, (display or {}).get(name, name))))\n    out = {"video": result["video"],\n           "thresholds": {labels.get(name, name): value for name, value in result["thresholds"].items()},\n           "segments": segments, "performance": result.get("performance", {})}\n    with open(json_path, "w", encoding="utf-8") as f:\n        json.dump(out, f, indent=2)\n\n\ndef plot_timeline(result, class_names, path: Optional[str] = None, display=None):\n    import matplotlib.pyplot as plt\n\n    sc = result["frame_scores"]\n    t = np.arange(len(sc)) / result["fps"]\n    fig, axes = plt.subplots(len(class_names), 1, figsize=(14, 2.4 * len(class_names)), sharex=True)\n    axes = np.atleast_1d(axes)\n    for c, (ax, name) in enumerate(zip(axes, class_names)):\n        ax.plot(t, sc[:, c], lw=0.8)\n        thr = result["thresholds"].get(name, 0.5)\n        ax.axhline(thr, color="r", ls="--", lw=0.8)\n        for s in result["segments"]:\n            if s.label == name:\n                ax.axvspan(s.start, s.end, color="orange", alpha=0.3)\n        ax.set_ylim(0, 1)\n        ax.set_ylabel((display or {}).get(name, name))\n    axes[-1].set_xlabel("time (s)")\n    plt.tight_layout()\n    if path:\n        plt.savefig(path, dpi=120)\n    return fig\n'}
runtime_dir = (Path("/content") if Path("/content").is_dir() else Path.cwd() / ".cache") / "video_event_runtime"
for relative, body in package_sources.items():
    target = runtime_dir / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(body, encoding="utf-8")
for module_name in list(sys.modules):
    if module_name == "ved" or module_name.startswith("ved."):
        del sys.modules[module_name]
if str(runtime_dir) in sys.path:
    sys.path.remove(str(runtime_dir))
sys.path.insert(0, str(runtime_dir))
importlib.invalidate_caches()
from huggingface_hub import hf_hub_download
from ved.inference import Detector, save_result, plot_timeline

checkpoint = local_checkpoint.strip()
if checkpoint:
    if not Path(checkpoint).expanduser().is_file():
        raise FileNotFoundError("Checkpoint lokal tidak ditemukan.")
    checkpoint = str(Path(checkpoint).expanduser().resolve())
else:
    token = os.environ.get("HF_TOKEN")
    if token is None:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            token = None
    checkpoint = hf_hub_download(model_repo, model_file, token=token)
names = [name.strip() for name in class_names_override.split(",")] if class_names_override.strip() else None
detector = Detector(checkpoint, device="cuda" if require_gpu else None, fp16=use_fp16,
                    class_names=names)
if stride_sec > detector.win / detector.fps:
    raise ValueError("Stride tidak boleh lebih panjang dari jendela model.")
print("Workflow: original labels, no archive, prefetch + adaptive scan")
print("Device:", detector.device, "| Kelas:", ", ".join(detector.class_names))
effective_thresholds = detector.resolve_thresholds(thresholds)
print("Threshold efektif:", json.dumps(effective_thresholds))
print("Threshold bawah:", json.dumps({name: value * low_ratio
                                      for name, value in effective_thresholds.items()}))
print("Stride:", stride_sec, "| Smooth:", smooth_sec,
      "| Merge gap:", merge_gap_sec, "| Durasi minimum:", min_duration_sec)
if any(value == 0 for value in effective_thresholds.values()):
    print("PERHATIAN: threshold 0 meloloskan semua skor. Gunakan -1 pada formulir untuk kalibrasi model.")
if detector.device.startswith("cuda"):
    print("GPU:", torch.cuda.get_device_name(0))

out = Path(output_dir).expanduser().resolve()
out.mkdir(parents=True, exist_ok=True)
source = video_url.strip()
if urlsplit(source).scheme.lower() in {"http", "https"}:
    if not shutil.which("wget"):
        if not Path("/content").is_dir():
            raise RuntimeError("Pasang wget dan tambahkan ke PATH terlebih dahulu.")
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "wget"], check=True)
    suffix = Path(urlsplit(source).path).suffix.lower()
    if suffix not in {".mp4", ".mkv", ".webm", ".mov", ".avi", ".m4v"}:
        suffix = ".mp4"
    video_path = out / ("input_video" + suffix)
    with tempfile.TemporaryDirectory(prefix=".download-", dir=out) as temp:
        temporary = Path(temp) / ("video" + suffix)
        print("Mengunduh video dengan wget...", flush=True)
        subprocess.run(["wget", "-O", str(temporary), "--", source], check=True)
        if not temporary.is_file() or temporary.stat().st_size == 0:
            raise RuntimeError("Unduhan tidak menghasilkan video.")
        os.replace(temporary, video_path)
else:
    video_path = Path(source).expanduser().resolve()
if not video_path.is_file() or video_path.stat().st_size == 0:
    raise FileNotFoundError("Video belum ditemukan atau kosong. Upload file ke panel Files.")

import cv2
from ved.video_io import video_info, export_segment
source_fps, source_frames, source_duration = video_info(str(video_path))
with open(checkpoint, "rb") as checkpoint_file:
    checkpoint_sha256 = hashlib.file_digest(checkpoint_file, "sha256").hexdigest()
run_config = {
    "checkpoint_sha256": checkpoint_sha256,
    "model_settings": detector.meta,
    "effective_thresholds": effective_thresholds,
    "threshold_overrides": thresholds,
    "settings": {"batch_size": int(batch_size), "stride_sec": stride_sec,
                 "prefetch": prefetch, "scan": scan_mode, "gate_ratio": gate_ratio,
                 "refine_padding": refine_padding_sec,
                 "analyze_from_sec": analyze_from_sec, "analyze_until_sec": analyze_until_sec,
                 "fp16": detector.fp16, "device": detector.device,
                 "agg": "mean", "smooth_sec": smooth_sec, "merge_gap": merge_gap_sec,
                 "min_duration": min_duration_sec, "low_ratio": low_ratio},
    "video": {"path": str(video_path), "bytes": video_path.stat().st_size,
              "fps": source_fps, "frames": source_frames, "duration": source_duration},
    "versions": {"torch": torch.__version__, "opencv": cv2.__version__},
}
config_path = out / "run_config.json"
config_path.write_text(json.dumps(run_config, indent=2), encoding="utf-8")
print("Checkpoint SHA256:", checkpoint_sha256)

result = detector.detect(
    str(video_path), batch_size=int(batch_size), stride_sec=stride_sec,
    thresholds=thresholds, smooth_sec=smooth_sec, merge_gap=merge_gap_sec,
    min_dur=min_duration_sec, low_ratio=low_ratio,
    prefetch=prefetch, scan=scan_mode, gate_ratio=gate_ratio,
    refine_padding=refine_padding_sec, start_sec=analyze_from_sec,
    end_sec=None if analyze_until_sec == -1 else analyze_until_sec,
)
print(detector.report(result))
paths = {"video": video_path, "json": out / "result.json",
         "config": config_path,
         "timeline": out / "timeline.png"}
save_result(result, str(paths["json"]), detector.display)
figure = plot_timeline(result, detector.class_names, str(paths["timeline"]), detector.display)
import matplotlib.pyplot as plt
plt.close(figure)
data = json.loads(paths["json"].read_text(encoding="utf-8"))
if result["segments"]:
    best_index = max(range(len(result["segments"])),
                     key=lambda index: (result["segments"][index].peak,
                                        result["segments"][index].mean))
    best = result["segments"][best_index]
    clip_path = out / "best_clip.mp4"
    if clip_path.resolve() == video_path.resolve():
        clip_path = out / "best_detection.mp4"
    export_segment(str(video_path), str(clip_path), best.start, best.end)
    paths["clip"] = clip_path
    data["best_clip"] = dict(data["segments"][best_index], file=str(clip_path))
    print("Klip terpilih:", clip_path, "| Confidence peak:", best.peak)
else:
    data["best_clip"] = None
    print("Tidak ada segmen yang lolos threshold; klip tidak dibuat.")
paths["json"].write_text(json.dumps(data, indent=2), encoding="utf-8")
print(json.dumps(data, indent=2))
print("Hasil tersimpan:", out)

from IPython.display import Image, Video, display
display(Image(filename=str(paths["timeline"])))
if preview_video and "clip" in paths:
    display(Video(str(paths["clip"]), embed=True))
if download_results:
    try:
        from google.colab import files
    except ImportError:
        print("Hasil siap:", paths["json"], paths.get("clip", "Tidak ada klip"))
    else:
        files.download(str(paths["json"]))
        if "clip" in paths:
            files.download(str(paths["clip"]))